# Gold-standard (Cell Ranger) — manuscript analysis (Figure 4c)

## Rationale (what this experiment is proving)

This experiment is a *bounded* gold-standard check that markets IDTrack’s **time-axis** semantics without turning into an accuracy benchmark.

Conceptually:

- The same raw sequencing dataset is processed with Cell Ranger references built from multiple Ensembl releases.
- Each release yields a slightly different feature space (different gene identifiers / versioning / merge-split history).
- IDTrack is used to map each per-release feature space into a single target release while explicitly reporting 1→0 / 1→1 / 1→n outcomes.

The marketing claim supported here is:

- Under a fixed snapshot boundary, time-travel mapping produces **consistent target feature sets** across starting releases, while preserving explicit ambiguity.

## Inputs

This notebook analyzes the gold-standard pipeline outputs produced by:
- `experiment_cellranger_idtrack/create_data.ipynb`

## Outputs

It generates a manuscript-ready figure (Fig 4c):
- `idtrack-manuscript/figures/fig_gold_standard_cellranger.pdf`
- `idtrack-manuscript/figures/fig_gold_standard_cellranger_extended.pdf` (optional multi-panel marketing figure)
- `idtrack-manuscript/figures/fig_gold_standard_assembly_axis.pdf` (optional; if multiple assemblies are available)

It also writes optional cache/diagnostic artefacts under the experiment cache directory.

Optional manuscript tables:
- `idtrack-manuscript/tables/gold_standard_cellranger_per_release.tex`
- `idtrack-manuscript/tables/gold_standard_assembly_axis_summary.csv`

## Environment variables
- `GOLD_STANDARD_ANNDATA_DIR`: directory containing the generated `.h5ad` files.
- `IDTRACK_LOCAL_REPO` (optional): shared IDTrack cache directory.

## Caching
- Per-release IDTrack conversion outputs are cached under `idtrack/docs/_notebooks/idtrack_cache/experiments/gold_standard_cellranger/`.
- If caches are missing, this notebook computes them (graph loading can be memory-intensive).

## Interpretation guide

- **Raw genes**: features emitted by Cell Ranger for a given Ensembl release.
- **Mapped targets**: union of target identifiers returned by IDTrack when mapping into the chosen target release.
- **1→0 fraction**: features that cannot be mapped into the target release under the configured snapshot (measurable loss).
- **Changed-only fraction**: signals drift; identifiers that change under time travel.



In [ ]:
from __future__ import annotations

import os
import re
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import sys

# Add experiments/src to sys.path (repo-relative; works from nested notebook dirs)
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    atomic_write_text,
    notebook_context,
    read_pickle,
    save_figure,
    write_pickle,
)

ctx = notebook_context('gold_standard_cellranger', start=REPO_ROOT)
plt.rcParams.update({'savefig.dpi': 300})

IDTRACK_LOCAL_REPO = ctx.idtrack_local_repo
CACHE_DIR = ctx.experiment_cache
MANUSCRIPT_FIGURES = ctx.manuscript_figures
MANUSCRIPT_TABLES = ctx.manuscript_tables

print('Repo root:', REPO_ROOT)
print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('CACHE_DIR:', CACHE_DIR)
print('MANUSCRIPT_FIGURES:', MANUSCRIPT_FIGURES)
print('MANUSCRIPT_TABLES:', MANUSCRIPT_TABLES)



In [ ]:
# -------------------- Configuration --------------------

ANNDATA_DIR_RAW = os.environ.get('GOLD_STANDARD_ANNDATA_DIR', '').strip()
ANNDATA_DIR = Path(ANNDATA_DIR_RAW).expanduser().resolve() if ANNDATA_DIR_RAW else None

# Make sure nested imports (and subprocesses) see the same location.
os.environ.setdefault('IDTRACK_LOCAL_REPO', str(IDTRACK_LOCAL_REPO))

# Target harmonization settings
TARGET_RELEASE = 114
FINAL_DATABASE = None  # keep as Ensembl gene backbone
STRATEGY = 'best'

print('ANNDATA_DIR:', ANNDATA_DIR)
print('TARGET_RELEASE:', TARGET_RELEASE)
print('FINAL_DATABASE:', FINAL_DATABASE)
print('STRATEGY:', STRATEGY)



In [ ]:
# -------------------- Discover .h5ad files --------------------

if not ANNDATA_DIR or not ANNDATA_DIR.exists():
    h5ads: list[Path] = []
    print('Set GOLD_STANDARD_ANNDATA_DIR to the folder containing the generated .h5ad files.')
else:
    h5ads = sorted(ANNDATA_DIR.glob('*.h5ad'))
    print('Found .h5ad files:', len(h5ads))
    print('First 10:', [p.name for p in h5ads[:10]])

pat = re.compile(r'^(?P<dataset>.+?)_(?P<assembly>[^_]+)_(?P<release>\d+)\.h5ad$')

rows = []
for p in h5ads:
    m = pat.match(p.name)
    if not m:
        continue
    rows.append(
        {
            'dataset': m.group('dataset'),
            'assembly': m.group('assembly'),
            'release': int(m.group('release')),
            'path': str(p),
        }
    )

files = pd.DataFrame(rows).sort_values(['dataset', 'assembly', 'release']).reset_index(drop=True)

if files.empty:
    print('No matching .h5ad files found (expected name pattern: <dataset>_<assembly>_<release>.h5ad).')
    selection = None
else:
    # Auto-select the dataset/assembly with the most releases (best evidence for the figure).
    grp = files.groupby(['dataset', 'assembly']).size().reset_index(name='n_files')
    best = grp.sort_values(['n_files', 'dataset', 'assembly'], ascending=[False, True, True]).iloc[0]
    selection = {'dataset': best['dataset'], 'assembly': best['assembly']}
    print('Auto-selected:', selection)

files


In [ ]:
# -------------------- Ensure per-release conversion caches exist --------------------

import pickle


def _safe_stem(s: str) -> str:
    return ''.join(c if c.isalnum() or c in {'-', '_'} else '_' for c in str(s))


def _conv_pickle_path(dataset: str, assembly: str, release: int) -> Path:
    return CACHE_DIR / (
        f"idtrack_matchings_{_safe_stem(dataset)}_{_safe_stem(assembly)}_from{release}_to{TARGET_RELEASE}"
        f"_final{FINAL_DATABASE or 'ensembl'}_strategy{STRATEGY}.pickle"
    )


def _raw_genes_pickle_path(dataset: str, assembly: str, release: int) -> Path:
    return CACHE_DIR / f"raw_genes_{_safe_stem(dataset)}_{_safe_stem(assembly)}_{release}.pickle"


@dataclass(frozen=True)
class ConversionPayload:
    matchings: list[dict]
    seconds: float
    n_inputs: int

    @property
    def it_per_s(self) -> float:
        return (self.n_inputs / self.seconds) if self.seconds else float('nan')


def _load_conv_payload(dataset: str, assembly: str, release: int) -> ConversionPayload | None:
    p = _conv_pickle_path(dataset, assembly, release)
    if not p.exists():
        return None

    obj = read_pickle(p)

    if isinstance(obj, list):
        return ConversionPayload(matchings=obj, seconds=float('nan'), n_inputs=len(obj))

    if not isinstance(obj, dict) or 'matchings' not in obj:
        raise TypeError(f"Unexpected payload in {p}: expected dict with 'matchings'.")

    return ConversionPayload(
        matchings=obj['matchings'],
        seconds=float(obj.get('seconds', float('nan'))),
        n_inputs=int(obj.get('n_inputs', len(obj['matchings']))),
    )


def _save_conv_payload(dataset: str, assembly: str, release: int, payload: ConversionPayload) -> Path:
    p = _conv_pickle_path(dataset, assembly, release)
    return write_pickle({'matchings': payload.matchings, 'seconds': payload.seconds, 'n_inputs': payload.n_inputs}, p)


def _load_raw_genes(dataset: str, assembly: str, release: int) -> set[str] | None:
    p = _raw_genes_pickle_path(dataset, assembly, release)
    if not p.exists():
        return None
    obj = read_pickle(p)
    return set(obj) if isinstance(obj, (list, set, tuple)) else None


def _save_raw_genes(dataset: str, assembly: str, release: int, genes: set[str]) -> Path:
    p = _raw_genes_pickle_path(dataset, assembly, release)
    return write_pickle(sorted(genes), p)


if files.empty or selection is None:
    print('No files to process.')
else:
    subset = files[(files['dataset'] == selection['dataset']) & (files['assembly'] == selection['assembly'])].copy()
    subset = subset.sort_values('release').reset_index(drop=True)

    dataset = str(selection['dataset'])
    assembly = str(selection['assembly'])

    # Identify which releases are missing caches
    missing = []
    for r in subset.itertuples(index=False):
        if _load_raw_genes(dataset, assembly, int(r.release)) is None:
            missing.append(('raw', int(r.release)))
        if _load_conv_payload(dataset, assembly, int(r.release)) is None:
            missing.append(('idtrack', int(r.release)))

    if not missing:
        print(f'All caches present for {dataset}/{assembly} (n={len(subset)} releases).')
    else:
        # Read raw genes (backed) and run IDTrack for missing conversions.
        import anndata as ad
        import idtrack

        api = idtrack.API(local_repository=str(IDTRACK_LOCAL_REPO))
        api.configure_logger()

        organism, _latest = api.resolve_organism('human')
        print(f'Building/loading IDTrack graph snapshot_release={TARGET_RELEASE} for {organism}')
        api.build_graph(organism_name=organism, snapshot_release=TARGET_RELEASE, calculate_caches=True)

        run_rows = []

        for r in subset.itertuples(index=False):
            rel = int(r.release)

            # Cache raw var_names
            raw_genes = _load_raw_genes(dataset, assembly, rel)
            if raw_genes is None:
                adata = ad.read_h5ad(r.path, backed='r')
                try:
                    raw_genes = set(adata.var_names.astype(str).tolist())
                finally:
                    try:
                        adata.file.close()
                    except Exception:
                        pass
                _save_raw_genes(dataset, assembly, rel, raw_genes)

            # Cache IDTrack conversion payload
            if _load_conv_payload(dataset, assembly, rel) is None:
                ids = sorted(raw_genes)
                t0 = time.perf_counter()
                matchings = api.convert_identifier_multiple(
                    ids,
                    to_release=TARGET_RELEASE,
                    final_database=FINAL_DATABASE,
                    strategy=STRATEGY,
                    verbose=True,
                    pbar_prefix=f"gold_standard:{dataset}:{assembly}:r{rel}",
                )
                dt = time.perf_counter() - t0
                payload = ConversionPayload(matchings=matchings, seconds=dt, n_inputs=len(ids))
                out_p = _save_conv_payload(dataset, assembly, rel, payload)
                print('Saved:', out_p.name)

                bins = api.classify_multiple_conversion(matchings)
                run_rows.append(
                    {
                        'release': rel,
                        'n_inputs': len(bins['input_identifiers']),
                        'n_1_to_0': len(bins['matching_1_to_0']),
                        'n_1_to_1': len(bins['matching_1_to_1']),
                        'n_1_to_n': len(bins['matching_1_to_n']),
                        'seconds': payload.seconds,
                        'it_per_s': payload.it_per_s,
                    }
                )

        if run_rows:
            runs = pd.DataFrame(run_rows).sort_values('release').reset_index(drop=True)
            out_csv = CACHE_DIR / f"conversion_runs_{_safe_stem(dataset)}_{_safe_stem(assembly)}.csv"
            atomic_write_text(out_csv, runs.to_csv(index=False))
            print('Wrote:', out_csv)
            runs


In [ ]:
# -------------------- Figure 4c: cross-release consistency under gold standard --------------------

from collections import OrderedDict

if files.empty or selection is None:
    print('No data available.')
else:
    dataset = str(selection['dataset'])
    assembly = str(selection['assembly'])
    subset = files[(files['dataset'] == dataset) & (files['assembly'] == assembly)].copy()
    subset = subset.sort_values('release').reset_index(drop=True)

    print('Dataset:', dataset)
    print('Assembly:', assembly)
    print('Releases:', subset['release'].tolist()[:10], '...' if len(subset) > 10 else '')

    # Load cached gene sets + conversions
    raw_sets: dict[int, set[str]] = {}
    conv_sets: dict[int, set[str]] = {}
    per_release_rows = []

    for r in subset.itertuples(index=False):
        rel = int(r.release)

        raw = _load_raw_genes(dataset, assembly, rel)
        payload = _load_conv_payload(dataset, assembly, rel)

        if raw is None or payload is None:
            raise RuntimeError(
                f"Missing caches for release {rel}. Run the notebook from the top to (re)create caches."
            )

        raw_sets[rel] = set(raw)

        # Strategy='best' should yield 1→1 targets, but keep a robust filter.
        mapped = set()
        n_failed = 0
        for rec in payload.matchings:
            if rec.get('no_corresponding') or rec.get('no_conversion'):
                n_failed += 1
                continue
            tids = rec.get('target_id', [])
            if isinstance(tids, list) and tids:
                mapped.add(str(tids[0]))

        conv_sets[rel] = mapped

        bins = api.classify_multiple_conversion(payload.matchings)
        per_release_rows.append(
            {
                'release': rel,
                'n_raw': len(raw_sets[rel]),
                'n_mapped': len(conv_sets[rel]),
                'n_failed_1_to_0': len(bins['matching_1_to_0']),
                'n_changed_only_1_to_1': len(bins['changed_only_1_to_1']),
            }
        )

    releases = list(subset['release'].astype(int))

    def _jaccard(a: set[str], b: set[str]) -> float:
        if not a and not b:
            return float('nan')
        inter = len(a & b)
        union = len(a | b)
        return inter / union if union else float('nan')

    raw_j = pd.DataFrame(index=releases, columns=releases, dtype=float)
    conv_j = pd.DataFrame(index=releases, columns=releases, dtype=float)

    for i in releases:
        for j in releases:
            raw_j.loc[i, j] = _jaccard(raw_sets[i], raw_sets[j])
            conv_j.loc[i, j] = _jaccard(conv_sets[i], conv_sets[j])

    stats = pd.DataFrame(per_release_rows).sort_values('release').reset_index(drop=True)
    out_stats = CACHE_DIR / f"gold_standard_summary_{_safe_stem(dataset)}_{_safe_stem(assembly)}.csv"
    atomic_write_text(out_stats, stats.to_csv(index=False))
    print('Wrote:', out_stats)

    fig = plt.figure(figsize=(12, 8), constrained_layout=True)
    gs = fig.add_gridspec(2, 2)
    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[0, 1])
    ax2 = fig.add_subplot(gs[1, 0])
    ax3 = fig.add_subplot(gs[1, 1])

    if sns is not None:
        sns.heatmap(raw_j, ax=ax0, vmin=0, vmax=1, cmap='Blues', cbar=False)
        ax0.set_title('Before harmonization (raw var_names)')

        sns.heatmap(conv_j, ax=ax1, vmin=0, vmax=1, cmap='Blues')
        ax1.set_title(f'After IDTrack → r{TARGET_RELEASE} (mapped gene IDs)')
    else:
        ax0.imshow(raw_j.values, vmin=0, vmax=1)
        ax0.set_title('Before harmonization')
        ax1.imshow(conv_j.values, vmin=0, vmax=1)
        ax1.set_title('After IDTrack harmonization')

    for ax in (ax0, ax1):
        ax.set_xlabel('Release')
        ax.set_ylabel('Release')

    # Gene-set sizes
    ax2.plot(stats['release'], stats['n_raw'], label='Raw var_names', color=MANUSCRIPT_COLORS['1→1'])
    ax2.plot(stats['release'], stats['n_mapped'], label='Mapped targets', color=MANUSCRIPT_COLORS['1→n'])
    ax2.set_xlabel('Release')
    ax2.set_ylabel('# genes')
    ax2.set_title('Gene-set size across releases')
    ax2.legend(frameon=True)

    # Drift/failure diagnostic
    frac_failed = stats['n_failed_1_to_0'] / stats['n_raw'].replace(0, np.nan)
    frac_changed = stats['n_changed_only_1_to_1'] / stats['n_raw'].replace(0, np.nan)

    ax3.plot(stats['release'], frac_failed, label='1→0 fraction', color=MANUSCRIPT_COLORS['1→0'])
    ax3.plot(stats['release'], frac_changed, label='Changed-only 1→1 fraction', color=MANUSCRIPT_COLORS['1→1'])
    ax3.set_xlabel('Release')
    ax3.set_ylabel('Fraction of raw genes')
    ax3.set_ylim(0, 1)
    ax3.set_title('Mapping diagnostics')
    ax3.legend(frameon=True)

    written = save_figure(fig, 'fig_gold_standard_cellranger.pdf', ctx, formats=('pdf',))
    print('Saved:', written['pdf'])

    # Cache a compact stats table for downstream notebooks / manuscript edits
    stats_path = CACHE_DIR / f"gold_standard_stats_{_safe_stem(dataset)}_{_safe_stem(assembly)}.csv"
    atomic_write_text(stats_path, stats.to_csv(index=False))
    print('Wrote:', stats_path)

    # Optional marketing figure: pairwise overlap across starting releases
    releases = sorted(conv_sets)
    if len(releases) >= 2:
        jacc = pd.DataFrame(index=releases, columns=releases, dtype=float)
        for r1 in releases:
            a = set(conv_sets[r1])
            for r2 in releases:
                b = set(conv_sets[r2])
                denom = len(a | b)
                jacc.loc[r1, r2] = (len(a & b) / denom) if denom else 1.0

        fig_h, ax_h = plt.subplots(1, 1, figsize=(6.2, 5.2))
        if sns is not None:
            sns.heatmap(jacc, ax=ax_h, cmap='Blues', vmin=0, vmax=1, square=True, cbar_kws={'label': 'Jaccard'})
        else:
            im = ax_h.imshow(jacc.values, cmap='Blues', vmin=0, vmax=1)
            fig_h.colorbar(im, ax=ax_h, label='Jaccard')
            ax_h.set_xticks(range(len(releases)))
            ax_h.set_yticks(range(len(releases)))
            ax_h.set_xticklabels(releases, rotation=30, ha='right')
            ax_h.set_yticklabels(releases)

        ax_h.set_title('Gold standard: mapped target-set overlap across starting releases')
        ax_h.set_xlabel('Starting release')
        ax_h.set_ylabel('Starting release')
        fig_h.tight_layout()

        written_h = save_figure(fig_h, 'fig_gold_standard_jaccard_heatmap.pdf', ctx, formats=('pdf',))
        print('Saved:', written_h['pdf'])


## Extended exports (marketing / manuscript options)

The main Fig 4c panel (`fig_gold_standard_cellranger.pdf`) is intentionally compact.
This section adds *optional* manuscript-facing exports derived from cached results:

- A per-release outcome/runtime summary table (CSV + LaTeX).
- A multi-panel figure with (i) outcome fractions, (ii) throughput, and (iii) pairwise overlap distributions.

These exports do **not** re-run Cell Ranger; they only post-process cached IDTrack conversions.



In [ ]:
# -------------------- Extended exports: outcomes + runtime + overlap summaries --------------------

from itertools import combinations

if files.empty or selection is None:
    print('No data available (no matching .h5ad files discovered).')
else:
    dataset = str(selection['dataset'])
    assembly = str(selection['assembly'])

    runs_csv = CACHE_DIR / f"conversion_runs_{_safe_stem(dataset)}_{_safe_stem(assembly)}.csv"
    stats_csv = CACHE_DIR / f"gold_standard_stats_{_safe_stem(dataset)}_{_safe_stem(assembly)}.csv"

    if not runs_csv.exists() or not stats_csv.exists():
        raise FileNotFoundError(
            'Missing cached summaries. Run this notebook once from the top to generate: '
            f"{runs_csv.name} and {stats_csv.name}."
        )

    runs = pd.read_csv(runs_csv)
    stats = pd.read_csv(stats_csv)

    merged = runs.merge(stats, on='release', how='left')
    merged = merged.sort_values('release').reset_index(drop=True)

    merged['frac_1_to_0'] = merged['n_1_to_0'] / merged['n_inputs'].replace(0, np.nan)
    merged['frac_1_to_1'] = merged['n_1_to_1'] / merged['n_inputs'].replace(0, np.nan)
    merged['frac_1_to_n'] = merged['n_1_to_n'] / merged['n_inputs'].replace(0, np.nan)

    merged['frac_failed_1_to_0'] = merged['n_failed_1_to_0'] / merged['n_raw'].replace(0, np.nan)
    merged['frac_changed_only_1_to_1'] = merged['n_changed_only_1_to_1'] / merged['n_raw'].replace(0, np.nan)

    # Export a compact per-release table (CSV + LaTeX)
    table = merged[
        ['release', 'n_inputs', 'frac_1_to_0', 'frac_1_to_1', 'frac_1_to_n', 'frac_changed_only_1_to_1', 'it_per_s']
    ].copy()

    table_csv = CACHE_DIR / f"gold_standard_cellranger_per_release_{_safe_stem(dataset)}_{_safe_stem(assembly)}.csv"
    atomic_write_text(table_csv, table.to_csv(index=False))
    print('Wrote:', table_csv)

    def _pct(x: float) -> str:
        return f"{100 * float(x):.1f}" if pd.notnull(x) else 'NA'

    rows = []
    for r in table.itertuples(index=False):
        rows.append(
            ' & '.join(
                [
                    str(int(r.release)),
                    str(int(r.n_inputs)),
                    _pct(r.frac_1_to_0),
                    _pct(r.frac_1_to_1),
                    _pct(r.frac_1_to_n),
                    _pct(r.frac_changed_only_1_to_1),
                    f"{float(r.it_per_s):.1f}" if pd.notnull(r.it_per_s) else 'NA',
                ]
            )
            + ' \\'
        )

    caption = (
        'Gold-standard Cell Ranger: per-release mapping outcomes when converting gene identifiers into '
        f'Ensembl r{TARGET_RELEASE} (dataset={dataset}, assembly={assembly}).'
    ).replace('_', r'\_')

    tex_lines = [
        r'\begin{table}[t]',
        r'\centering',
        f'\caption{{{caption}}}',
        r'\label{tab:gold-standard-cellranger-per-release}',
        r'\begin{tabular}{rrrrrrr}',
        r'\toprule',
        r'Release & $n$ & 1$\to$0 (\%) & 1$\to$1 (\%) & 1$\to$n (\%) & changed-only 1$\to$1 (\%) & it/s \\',
        r'\midrule',
        *rows,
        r'\bottomrule',
        r'\end{tabular}',
        r'\end{table}',
    ]

    out_tex = MANUSCRIPT_TABLES / 'gold_standard_cellranger_per_release.tex'
    tex = '\n'.join(tex_lines) + '\n'
    atomic_write_text(out_tex, tex)
    atomic_write_text((ctx.experiment_outputs / 'tables' / out_tex.name), tex)
    print('Wrote:', out_tex)
    print('Wrote:', (ctx.experiment_outputs / 'tables' / out_tex.name))

    # Pairwise overlap distributions (raw vs mapped target sets)
    subset = files[(files['dataset'] == dataset) & (files['assembly'] == assembly)].sort_values('release')
    releases = [int(x) for x in subset['release'].tolist()]

    raw_sets: dict[int, set[str]] = {}
    conv_sets: dict[int, set[str]] = {}

    for rel in releases:
        raw = _load_raw_genes(dataset, assembly, rel)
        payload = _load_conv_payload(dataset, assembly, rel)
        if raw is None or payload is None:
            raise RuntimeError(f'Missing cached payload for release {rel}.')

        raw_sets[rel] = set(raw)
        mapped = set()
        for rec in payload.matchings:
            if rec.get('no_corresponding') or rec.get('no_conversion') or rec.get('no_target'):
                continue
            tids = rec.get('target_id', [])
            if isinstance(tids, list) and tids:
                mapped.add(str(tids[0]))
        conv_sets[rel] = mapped

    def _jaccard(a: set[str], b: set[str]) -> float:
        denom = len(a | b)
        return (len(a & b) / denom) if denom else 1.0

    pair_rows = []
    for r1, r2 in combinations(releases, 2):
        pair_rows.append(
            {
                'r1': r1,
                'r2': r2,
                'distance': abs(int(r1) - int(r2)),
                'raw_jaccard': _jaccard(raw_sets[r1], raw_sets[r2]),
                'mapped_jaccard': _jaccard(conv_sets[r1], conv_sets[r2]),
            }
        )

    pairs = pd.DataFrame(pair_rows)
    if not pairs.empty:
        pairs['delta'] = pairs['mapped_jaccard'] - pairs['raw_jaccard']

    out_pairs = CACHE_DIR / f"gold_standard_jaccard_pairs_{_safe_stem(dataset)}_{_safe_stem(assembly)}.csv"
    atomic_write_text(out_pairs, pairs.to_csv(index=False))
    print('Wrote:', out_pairs)

    # Multi-panel marketing figure
    fig, axes = plt.subplots(2, 2, figsize=(12.6, 8.0), constrained_layout=True)
    ax0, ax1, ax2, ax3 = axes.ravel()

    # A) outcome fractions across starting releases
    ax0.stackplot(
        merged['release'],
        merged['frac_1_to_0'],
        merged['frac_1_to_1'],
        merged['frac_1_to_n'],
        labels=['1→0', '1→1', '1→n'],
        colors=[MANUSCRIPT_COLORS['1→0'], MANUSCRIPT_COLORS['1→1'], MANUSCRIPT_COLORS['1→n']],
        alpha=0.9,
    )
    ax0.set_ylim(0, 1)
    ax0.set_xlabel('Starting release')
    ax0.set_ylabel('Fraction of queries')
    ax0.set_title('A) Outcome fractions vs release')
    ax0.legend(frameon=True, loc='lower left')

    # B) throughput
    ax1.plot(merged['release'], merged['it_per_s'], '-o', color=MANUSCRIPT_COLORS['neutral'], lw=1.4, ms=3)
    ax1.set_xlabel('Starting release')
    ax1.set_ylabel('Identifiers / s')
    ax1.set_title('B) Conversion throughput')

    # C) feature-space size before/after mapping
    ax2.plot(merged['release'], merged['n_raw'], '-o', label='Raw genes', color=MANUSCRIPT_COLORS['1→1'], lw=1.4, ms=3)
    ax2.plot(merged['release'], merged['n_mapped'], '-o', label='Mapped targets', color=MANUSCRIPT_COLORS['1→n'], lw=1.4, ms=3)
    ax2.set_xlabel('Starting release')
    ax2.set_ylabel('# genes')
    ax2.set_title('C) Feature-space size')
    ax2.legend(frameon=True)

    # D) pairwise overlap distributions
    if pairs.empty:
        ax3.axis('off')
    else:
        ax3.hist(pairs['raw_jaccard'], bins=18, alpha=0.65, label='Raw', color=MANUSCRIPT_COLORS['1→1'])
        ax3.hist(pairs['mapped_jaccard'], bins=18, alpha=0.55, label='Mapped', color=MANUSCRIPT_COLORS['1→n'])
        ax3.set_xlabel('Jaccard overlap')
        ax3.set_ylabel('# pairs')
        ax3.set_title('D) Pairwise overlap (off-diagonal)')
        ax3.legend(frameon=True)

    written = save_figure(fig, 'fig_gold_standard_cellranger_extended.pdf', ctx, formats=('pdf',))
    print('Saved:', written['pdf'])

    pairs.head() if not pairs.empty else None



# Marketing extension: assembly axis (when available)

IDTrack’s framing emphasizes that identifier identity depends on **namespace × release × assembly**.

If your gold-standard dataset was built across multiple genome assemblies (e.g., GRCh37 vs GRCh38),
this section summarizes how much the shared feature-space overlap improves after harmonization *per assembly*.

This is a marketing-friendly way to justify “assembly-aware mapping” without overclaiming accuracy.


In [ ]:
from experiments_utils import atomic_write_dataframe_csv  # noqa: E402

if files.empty or selection is None:
    print('No discovered gold-standard files; skipping assembly analysis.')
else:
    dataset = str(selection['dataset'])
    assemblies = sorted(set(files[files['dataset'] == dataset]['assembly'].astype(str).tolist()))
    if len(assemblies) < 2:
        print('Only one assembly detected for the selected dataset; nothing to compare.')
    else:
        rows = []

        def _mean_offdiag_jaccard(sets: dict[int, set[str]]) -> float:
            rels = sorted(sets)
            if len(rels) < 2:
                return float('nan')
            vals = []
            for i, r1 in enumerate(rels):
                for r2 in rels[i + 1:]:
                    a = sets.get(r1, set())
                    b = sets.get(r2, set())
                    denom = len(a | b)
                    vals.append((len(a & b) / denom) if denom else float('nan'))
            return float(np.nanmean(vals)) if vals else float('nan')

        for assembly in assemblies:
            subset = files[(files['dataset'] == dataset) & (files['assembly'] == assembly)].copy()
            subset = subset.sort_values('release').reset_index(drop=True)
            if subset.empty:
                continue

            raw_sets = {}
            conv_sets = {}
            ok = True
            for r in subset.itertuples(index=False):
                rel = int(r.release)
                raw = _load_raw_genes(dataset, assembly, rel)
                payload = _load_conv_payload(dataset, assembly, rel)
                if raw is None or payload is None:
                    ok = False
                    break
                raw_sets[rel] = set(raw)
                mapped = set()
                for rec in payload.matchings:
                    if rec.get('no_corresponding') or rec.get('no_conversion'):
                        continue
                    tids = rec.get('target_id', [])
                    if isinstance(tids, list) and tids:
                        mapped.add(str(tids[0]))
                conv_sets[rel] = mapped

            if not ok:
                print('Skipping assembly (missing caches):', assembly)
                continue

            rows.append(
                {
                    'dataset': dataset,
                    'assembly': assembly,
                    'n_releases': int(len(raw_sets)),
                    'mean_raw_offdiag_jaccard': _mean_offdiag_jaccard(raw_sets),
                    'mean_mapped_offdiag_jaccard': _mean_offdiag_jaccard(conv_sets),
                }
            )

        asm = pd.DataFrame(rows)
        if asm.empty:
            print('No assemblies had complete caches; skipping export.')
        else:
            asm['delta_mapped_minus_raw'] = asm['mean_mapped_offdiag_jaccard'] - asm['mean_raw_offdiag_jaccard']
            out_asm = MANUSCRIPT_TABLES / 'gold_standard_assembly_axis_summary.csv'
            atomic_write_dataframe_csv(asm, out_asm, index=False)
            atomic_write_dataframe_csv(asm, ctx.experiment_outputs / 'tables' / out_asm.name, index=False)
            print('Wrote:', out_asm)
            display(asm)

            figA, axA = plt.subplots(1, 1, figsize=(7.4, 3.8), constrained_layout=True)
            axA.bar(asm['assembly'], asm['delta_mapped_minus_raw'], color=MANUSCRIPT_COLORS['1→1'])
            axA.axhline(0, color=MANUSCRIPT_COLORS['grid'], lw=1)
            axA.set_ylabel('Δ mean off-diagonal Jaccard (mapped − raw)')
            axA.set_xlabel('assembly')
            axA.set_title('Assembly axis: harmonization increases cross-release consistency')
            writtenA = save_figure(figA, 'fig_gold_standard_assembly_axis.pdf', ctx, formats=('pdf',))
            print('Saved:', writtenA['pdf'])
